https://bit.ly/DSNN-1-intro

# Imbalanced classification: credit card fraud detection

## Introduction

This example looks at the
[Kaggle Credit Card Fraud Detection](https://www.kaggle.com/mlg-ulb/creditcardfraud/)
dataset to demonstrate how
to train a classification model on data with highly imbalanced classes.

## First, vectorize the CSV data

In [1]:
!wget https://www.dropbox.com/s/9vfy1vi6wsfkxyk/creditcard.csv.zip

"wget" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [2]:
!unzip creditcard.csv.zip

"unzip" �� ���� ����७��� ��� ���譥�
��������, �ᯮ��塞�� �ணࠬ��� ��� ������ 䠩���.


In [3]:
import pandas as pd
import numpy as np

data = pd.read_csv('creditcard.csv')
data.head()

FileNotFoundError: [Errno 2] No such file or directory: 'creditcard.csv'

In [ ]:
features = data.drop(columns=['Class']).values
targets = data[['Class']].values.astype('uint8')

In [ ]:
features.shape

(284807, 30)

In [ ]:
targets.shape

(284807, 1)

In [ ]:
targets

array([[0],
       [0],
       [0],
       ...,
       [0],
       [0],
       [0]], dtype=uint8)

In [ ]:
features

array([[ 0.00000000e+00, -1.35980713e+00, -7.27811733e-02, ...,
         1.33558377e-01, -2.10530535e-02,  1.49620000e+02],
       [ 0.00000000e+00,  1.19185711e+00,  2.66150712e-01, ...,
        -8.98309914e-03,  1.47241692e-02,  2.69000000e+00],
       [ 1.00000000e+00, -1.35835406e+00, -1.34016307e+00, ...,
        -5.53527940e-02, -5.97518406e-02,  3.78660000e+02],
       ...,
       [ 1.72788000e+05,  1.91956501e+00, -3.01253846e-01, ...,
         4.45477214e-03, -2.65608286e-02,  6.78800000e+01],
       [ 1.72788000e+05, -2.40440050e-01,  5.30482513e-01, ...,
         1.08820735e-01,  1.04532821e-01,  1.00000000e+01],
       [ 1.72792000e+05, -5.33412522e-01, -1.89733337e-01, ...,
        -2.41530880e-03,  1.36489143e-02,  2.17000000e+02]])

## Prepare a validation set

In [ ]:
num_val_samples = int(len(features) * 0.2)
train_features = features[:-num_val_samples]
train_targets = targets[:-num_val_samples]
val_features = features[-num_val_samples:]
val_targets = targets[-num_val_samples:]

print("Number of training samples:", len(train_features))
print("Number of validation samples:", len(val_features))

Number of training samples: 227846
Number of validation samples: 56961


## Analyze class imbalance in the targets

In [ ]:
counts = np.bincount(train_targets[:, 0])
counts

array([227429,    417])

In [ ]:
print(
    f"Number of positive samples in training data: {counts[1]} ({100 * float(counts[1]) / len(train_targets):.2f}% of total)"
)

weight_for_0 = 1.0 / counts[0]
weight_for_1 = 1.0 / counts[1]

Number of positive samples in training data: 417 (0.18% of total)


## Normalize the data using training set statistics

In [ ]:
mean = np.mean(train_features, axis=0)
train_features -= mean
val_features -= mean

std = np.std(train_features, axis=0)
train_features /= std
val_features /= std

## Build a binary classification model

### Dropout

Метод регуляризации искусственных нейронных сетей, предназначен для уменьшения переобучения сети за счет предотвращения сложных адаптаций отдельных нейронов на тренировочных данных во время обучения.

Характеризует исключение определённого процента (например 50%) случайных нейронов на разных итерациях во время обучения нейронной сети. В результате  обучение происходит более общее, нет надежды на определенные нейроны. Такой приём значительно увеличивает скорость обучения, качество обучения на тренировочных данных, а также повышает качество предсказаний модели на новых тестовых данных.

На моменте предсказания все нейроны включаются обратно, dropout не используется.

<img src='https://drive.google.com/uc?export=view&id=1KQrdTDanDkLhf8Kn8c5ryjqN2acwYWII' width=500>

In [ ]:
from tensorflow import keras

hid_size = 256
model = keras.Sequential(
    [
        keras.layers.Dense(
            hid_size, activation="relu", input_shape=(train_features.shape[-1],)
        ), # fully-connected y^1
        keras.layers.Dense(hid_size, activation="relu"), # y^2
        keras.layers.Dropout(0.3),
        keras.layers.Dense(hid_size, activation="relu"), # y^3
        keras.layers.Dropout(0.3),
        keras.layers.Dense(1, activation="sigmoid"), # y^4
    ]
)
model.summary()

Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_8 (Dense)             (None, 256)               7936      
                                                                 
 dense_9 (Dense)             (None, 256)               65792     
                                                                 
 dropout_4 (Dropout)         (None, 256)               0         
                                                                 
 dense_10 (Dense)            (None, 256)               65792     
                                                                 
 dropout_5 (Dropout)         (None, 256)               0         
                                                                 
 dense_11 (Dense)            (None, 1)                 257       
                                                                 
Total params: 139,777
Trainable params: 139,777
Non-tr

## Train the model with `class_weight` argument

In [ ]:
metrics = [
    keras.metrics.FalseNegatives(name="fn"),
    keras.metrics.FalsePositives(name="fp"),
    keras.metrics.Precision(name="precision"),
    keras.metrics.Recall(name="recall"),
]

model.compile(
    optimizer=keras.optimizers.Adam(1e-2),
    loss="binary_crossentropy",
    metrics=metrics
)

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint("fraud_model_at_epoch_{epoch}.h5")
]

class_weight = {0: weight_for_0, 1: weight_for_1}



<img src='https://drive.google.com/uc?export=view&id=1j8SxKYEi12jzJXi_bPO28q5SV9emuu0Y'>

In [ ]:
model.fit(
    train_features,
    train_targets,
    batch_size=2048,
    epochs=30,
    callbacks=callbacks,
    validation_data=(val_features, val_targets),
    class_weight=class_weight,
)


Epoch 1/30
112/112 [==============================] - 9s 65ms/step - loss: 2.3283e-06 - fn: 47.0000 - fp: 25104.0000 - precision: 0.0145 - recall: 0.8873 - val_loss: 0.0848 - val_fn: 9.0000 - val_fp: 1110.0000 - val_precision: 0.0561 - val_recall: 0.8800
Epoch 2/30
112/112 [==============================] - 7s 60ms/step - loss: 1.4262e-06 - fn: 32.0000 - fp: 6475.0000 - precision: 0.0561 - recall: 0.9233 - val_loss: 0.0658 - val_fn: 13.0000 - val_fp: 200.0000 - val_precision: 0.2366 - val_recall: 0.8267
Epoch 3/30
112/112 [==============================] - 7s 60ms/step - loss: 1.3120e-06 - fn: 31.0000 - fp: 7364.0000 - precision: 0.0498 - recall: 0.9257 - val_loss: 0.0620 - val_fn: 8.0000 - val_fp: 872.0000 - val_precision: 0.0714 - val_recall: 0.8933
Epoch 4/30
112/112 [==============================] - 7s 59ms/step - loss: 1.1036e-06 - fn: 27.0000 - fp: 8056.0000 - precision: 0.0462 - recall: 0.9353 - val_loss: 0.0414 - val_fn: 10.0000 - val_fp: 622.0000 - val_precision: 0.0946 - val

#**Дополнительные материалы**


**По теории:**
1. [Interactive neural network playground in your browser](http://playground.tensorflow.org/#activation=tanh&batchSize=10&dataset=circle&regDataset=reg-plane&learningRate=0.03&regularizationRate=0&noise=0&networkShape=4,2&seed=0.72732&showTestData=false&discretize=false&percTrainData=50&x=true&y=true&xTimesY=false&xSquared=false&ySquared=false&cosX=false&sinX=false&cosY=false&sinY=false&collectStats=false&problem=classification&initZero=false&hideText=false)
2. Backprop in depth by cs231 - https://cs231n.github.io/optimization-2/
3. Минусы и плюсы функций активаций (видео) - https://youtu.be/Gs8T_qF-FAA
4. A recipe for training neural networks  - http://karpathy.github.io/2019/04/25/recipe/
5. Метод обратного распространения ошибки (видео) - https://youtu.be/EuhoXsuu8SQ
6. Примерно аналогичный материал с лекции, только от Стенфорда и на английском - http://cs231n.github.io/neural-networks-1/#nn
7. Official intro to Keras - https://keras.io/getting_started/intro_to_keras_for_researchers/
8. Deep learning frameworks (видео) - https://www.youtube.com/watch?v=ghZyptkanB0


**По практике:**
1. Метод fit в Keras (видео) - https://youtu.be/PLlic60dgS4
2. Callbacks в Keras. ModelCheckpoint, ReduceLROnPlateau, EarlyStopping (видео) - https://youtu.be/sq9HvLAsK94